In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from os import listdir
from itertools import permutations
import numpy as np
import matplotlib.pyplot as plt
from oxDNA_analysis_tools.oxDNA_PDB import get_nucs_from_PDB
from oxDNA_analysis_tools.UTILS.utils import kabsch_align
import oxpy

In [3]:
%matplotlib qt 

In [4]:
# Get the idealized oxDNA geometry
from oxDNA_analysis_tools.UTILS.utils import get_pos_back, get_pos_base, get_pos_stack
from oxDNA_analysis_tools.UTILS.RyeReader import describe, get_confs
ti, di = describe(None, '1p.dat')
conf = get_confs(ti, di, 0, 1)[0]
bbs_RNA = (get_pos_back(conf.positions[0], conf.a1s[0], conf.a3s[0], type='RNA') - conf.positions[0]) * 8.518
ss_RNA = (get_pos_stack(conf.positions[0], conf.a1s[0], conf.a3s[0], type='RNA') - conf.positions[0]) * 8.518
bs_RNA = (get_pos_base(conf.positions[0], conf.a1s[0], conf.a3s[0], type='RNA') - conf.positions[0]) * 8.518

bbs_DNA = (get_pos_back(conf.positions[0], conf.a1s[0], conf.a3s[0], type='DNA') - conf.positions[0]) * 8.518
ss_DNA = (get_pos_stack(conf.positions[0], conf.a1s[0], conf.a3s[0], type='DNA') - conf.positions[0]) * 8.518
bs_DNA = (get_pos_base(conf.positions[0], conf.a1s[0], conf.a3s[0], type='DNA') - conf.positions[0]) * 8.518

In [5]:
print("DNA")
print(bbs_DNA)
print(ss_DNA)
print(bs_DNA)
print()
print("RNA")
print(bbs_RNA)
print(ss_RNA)
print(bs_RNA)

DNA
[-2.89612    2.9029344  0.       ]
[2.89612 0.      0.     ]
[3.4072 0.     0.    ]

RNA
[-3.4072  0.      1.7036]
[2.89612 0.      0.     ]
[3.4072 0.     0.    ]


In [6]:
# Get the options from the PDB files
nucs = []
pdbfiles = [f for f in listdir() if '.pdb' in f]
for f in pdbfiles:
    nucs += get_nucs_from_PDB(f)

DNA_nucs = [n for n in nucs if 'D' in n.name]
RNA_nucs = [n for n in nucs if 'D' not in n.name]

In [7]:
candidates = {
    "base_centers" : lambda nuc: np.mean([a.pos - nuc.get_com() for a in nuc.base_atoms], axis=0),
    "sugar_centers" : lambda nuc: np.mean([a.pos - nuc.get_com() for a in nuc.sugar_atoms], axis=0),
    "base_edge" : lambda nuc: nuc['N1'].pos - nuc.get_com() if any((c in nuc.name for c in ['A', 'G'])) else nuc['N3'].pos - nuc.get_com(),
    "base_hex_back" : lambda nuc: nuc['C4'].pos - nuc.get_com() if any((c in nuc.name for c in ['A', 'G'])) else nuc['C6'].pos - nuc.get_com(),
    "base_hex_bottom" : lambda nuc: nuc['N3'].pos - nuc.get_com() if any((c in nuc.name for c in ['A', 'G'])) else nuc['C2'].pos - nuc.get_com(),
    "base_C2" : lambda nuc: nuc["C2"].pos - nuc.get_com(),
    "O3s" : lambda nuc: nuc["O3'"].pos - nuc.get_com(),
    "C3s" : lambda nuc: nuc["C3'"].pos - nuc.get_com(),
    "C4s" : lambda nuc: nuc["C4'"].pos - nuc.get_com(),
    "C5s" : lambda nuc: nuc["C5'"].pos - nuc.get_com(),
    "O5s" : lambda nuc: nuc["O5'"].pos - nuc.get_com(),
    "phos" : lambda nuc: nuc['P'].pos - nuc.get_com() if not '5' in nuc.name \
                    else nuc["HO5'"].pos - nuc.get_com() if "HO5'" in nuc.named_atoms \
                    else nuc["O5'"].pos - nuc.get_com() if "O5'" in nuc.named_atoms
                    else np.array([np.nan, np.nan, np.nan]), # This is hard because not all files have hydrogens
}
possibilities = list(permutations(candidates.keys(), 2))

In [8]:
results_DNA = {p : [] for p in possibilities}
ref_DNA = np.array([
    conf.a1s[0],
    conf.a3s[0],
    np.cross(conf.a3s[0], conf.a1s[0]),
    bbs_DNA,
    bs_DNA
])
for n in DNA_nucs:
    n.compute_as()
    for p in possibilities:
        points = np.array([
            n.a1,
            n.a3,
            n.a2,
            candidates[p[0]](n),
            candidates[p[1]](n)
        ])
        kabsch_align(points, ref_DNA, center=False, inplace=True)
        diff = np.mean(np.linalg.norm(points - ref_DNA, axis=1))
        results_DNA[p].append(diff)

In [9]:
means_DNA = np.array([np.mean(v) for v in results_DNA.values()])
stdevs_DNA = np.array([np.std(v) for v in results_DNA.values()])
min10_DNA = means_DNA.argsort()[:10]
for i in min10_DNA:
    print(possibilities[i])
    print(means_DNA[i], stdevs_DNA[i])

('O3s', 'base_centers')
0.4435724189890738 0.16981492555039074
('O3s', 'base_edge')
0.49563892813044336 0.15823996342320015
('O3s', 'base_C2')
0.5919426993877784 0.15122822889948379
('C4s', 'base_centers')
0.6192817601159301 0.14636514018168045
('O3s', 'base_hex_bottom')
0.6211178208592879 0.15227478080903098
('C4s', 'base_edge')
0.6229955251141549 0.17781731183731656
('C3s', 'base_edge')
0.6298785816759418 0.16531157181806297
('C5s', 'base_hex_bottom')
0.6301856233573034 0.18913583677309673
('C4s', 'base_C2')
0.6304296106085916 0.162834114375793
('C4s', 'base_hex_bottom')
0.6320863734560802 0.15674369005799801


In [10]:
best_DNA = possibilities[min10_DNA[0]]
print(best_DNA)
back_proxy = np.array([candidates[best_DNA[0]](n) for n in DNA_nucs])
base_proxy = np.array([candidates[best_DNA[1]](n) for n in DNA_nucs])
for i, nuc in enumerate(DNA_nucs):
    nuc.compute_as()
    points = np.array([
        nuc.a1,
        nuc.a3,
        nuc.a2,
        back_proxy[i],
        base_proxy[i]
    ])
    rot = kabsch_align(points, ref_DNA, center=False, inplace=True, return_rot=True)
    base_proxy[i] = np.dot(base_proxy[i], rot)
    back_proxy[i] = np.dot(back_proxy[i], rot)

    # poses = np.array([a.pos - nuc.get_com() for a in nuc.get_atoms()])
    # poses = np.dot(poses, rot)
    # ax = plt.figure().add_subplot(projection='3d')
    # Cid = [i for i, a in enumerate(nuc.get_atoms()) if 'C' in a.name]
    # Nid = [i for i, a in enumerate(nuc.get_atoms()) if 'N' in a.name]
    # Oid = [i for i, a in enumerate(nuc.get_atoms()) if a.name.startswith('O')]
    # Pid = [i for i, a in enumerate(nuc.get_atoms()) if a.name == 'P']
    # Hid = [i for i, a in enumerate(nuc.get_atoms()) if a.name.startswith('H')]
    # ax.scatter(poses[Cid][:,0], poses[Cid][:,1], poses[Cid][:,2], c='black', label='All-atom C')
    # ax.scatter(poses[Nid][:,0], poses[Nid][:,1], poses[Nid][:,2], c='blue', label='All-atom N')
    # ax.scatter(poses[Oid][:,0], poses[Oid][:,1], poses[Oid][:,2], c='red', label='All-atom O')
    # ax.scatter(poses[Pid][:,0], poses[Pid][:,1], poses[Pid][:,2], c='orange', label='All-atom P')
    # ax.scatter(poses[Hid][:,0], poses[Hid][:,1], poses[Hid][:,2], c='gray', label='All-atom H')
    # ax.quiver(np.zeros(3), np.zeros(3), np.zeros(3), ref_DNA[:3, 0], ref_DNA[:3, 1], ref_DNA[:3, 2])
    # ax.scatter(ref_DNA[3:,0], ref_DNA[3:,1], ref_DNA[3:,2], c='green', label='oxDNA')
    # ax.set_aspect('equal')
    # plt.show()
    
ax = plt.figure().add_subplot(projection='3d')
ax.scatter(base_proxy[:,0], base_proxy[:,1], base_proxy[:,2], c='lightskyblue', s=3, alpha=0.3, label=f'All-atom {best_DNA[1]}')
ax.scatter(back_proxy[:,0], back_proxy[:,1], back_proxy[:,2], c='wheat', s=3, alpha=0.3, label=f'All-atom {best_DNA[0]}')

ax.scatter(bs_DNA[0], bs_DNA[1], bs_DNA[2], c='blue', label='oxDNA base site')
ax.scatter(bbs_DNA[0], bbs_DNA[1], bbs_DNA[2], c='red', label='oxDNA back site')
ax.set_aspect('equal')

ax.legend()

('O3s', 'base_centers')


In [11]:
results = {p : [] for p in possibilities}
ref_RNA = np.array([
    conf.a1s[0],
    conf.a3s[0],
    np.cross(conf.a3s[0], conf.a1s[0]),
    bbs_RNA ,
    bs_RNA 
])
for n in RNA_nucs:
    n.compute_as()
    for p in possibilities:
        points = np.array([
            n.a1,
            n.a3,
            n.a2,
            candidates[p[0]](n),
            candidates[p[1]](n)
        ])
        kabsch_align(points, ref_RNA, center=False, inplace=True)
        diff = np.mean(np.linalg.norm(points - ref_RNA, axis=1))
        results[p].append(diff)

In [12]:
means = np.array([np.mean(v) for v in results.values()])
stdevs = np.array([np.std(v) for v in results.values()])
min10_RNA = means.argsort()[:10]
for i in min10_RNA:
    print(possibilities[i])
    print(means[i], stdevs[i])

('C5s', 'base_edge')
0.5026316569278846 0.07738943095775615
('C5s', 'base_centers')
0.5053038976797393 0.10269214955552652
('C5s', 'base_hex_bottom')
0.5361668958084672 0.04455481055068906
('C5s', 'base_C2')
0.5411686594240532 0.04419591573964326
('C4s', 'base_centers')
0.6062017333629461 0.0786325224830502
('O5s', 'base_hex_bottom')
0.6274908230579769 0.10101661982546749
('O5s', 'base_C2')
0.6630452963180256 0.07238466722153426
('phos', 'base_hex_bottom')
0.6734289534236441 0.09314724752113718
('C4s', 'base_edge')
0.6775628950000327 0.06104806767805495
('C5s', 'base_hex_back')
0.7126862650726267 0.19322343576134465


In [13]:
best_RNA = possibilities[min10_RNA[0]]
back_proxy = np.array([candidates[best_RNA[0]](n) for n in RNA_nucs])
base_proxy = np.array([candidates[best_RNA[1]](n) for n in RNA_nucs])
for i, nuc in enumerate(RNA_nucs):
    nuc.compute_as()
    points = np.array([
        nuc.a1,
        nuc.a3,
        nuc.a2,
        back_proxy[i],
        base_proxy[i]
    ])
    rot = kabsch_align(points, ref_RNA, center=False, inplace=True, return_rot=True)
    #com_proxy[i] = np.dot(com_proxy[i], rot)
    base_proxy[i] = np.dot(base_proxy[i], rot)
    #stack_proxy[i] = np.dot(stack_proxy[i], rot)
    back_proxy[i] = np.dot(back_proxy[i], rot)
    
    # poses = np.array([a.pos - nuc.get_com() for a in nuc.get_atoms()])
    # poses = np.dot(poses, rot)
    # ax = plt.figure().add_subplot(projection='3d')
    # Cid = [i for i, a in enumerate(nuc.get_atoms()) if 'C' in a.name]
    # Nid = [i for i, a in enumerate(nuc.get_atoms()) if 'N' in a.name]
    # Oid = [i for i, a in enumerate(nuc.get_atoms()) if a.name.startswith('O')]
    # Pid = [i for i, a in enumerate(nuc.get_atoms()) if a.name == 'P']
    # Hid = [i for i, a in enumerate(nuc.get_atoms()) if a.name.startswith('H')]
    # ax.scatter(poses[Cid][:,0], poses[Cid][:,1], poses[Cid][:,2], c='black', label='All-atom C')
    # ax.scatter(poses[Nid][:,0], poses[Nid][:,1], poses[Nid][:,2], c='blue', label='All-atom N')
    # ax.scatter(poses[Oid][:,0], poses[Oid][:,1], poses[Oid][:,2], c='red', label='All-atom O')
    # ax.scatter(poses[Pid][:,0], poses[Pid][:,1], poses[Pid][:,2], c='orange', label='All-atom P')
    # ax.scatter(poses[Hid][:,0], poses[Hid][:,1], poses[Hid][:,2], c='gray', label='All-atom H')
    # ax.quiver(np.zeros(3), np.zeros(3), np.zeros(3), ref_RNA[:3, 0], ref_RNA[:3, 1], ref_RNA[:3, 2])
    # ax.scatter(ref_RNA[3:,0], ref_RNA[3:,1], ref_RNA[3:,2], c='green', label='oxRNA')
    # ax.set_aspect('equal')
    # plt.show()
    
ax = plt.figure().add_subplot(projection='3d')
ax.scatter(base_proxy[:,0], base_proxy[:,1], base_proxy[:,2], c='lightskyblue', s=3, alpha=0.3, label=f'All-atom {best_RNA[1]}')
#ax.scatter(stack_proxy[:,0], stack_proxy[:,1], stack_proxy[:,2], c='#9ed870', s=3, alpha=0.3, label='All-atom base center')
ax.scatter(back_proxy[:,0], back_proxy[:,1], back_proxy[:,2], c='wheat', s=3, alpha=0.3, label=f'All-atom {best_RNA[0]}')

ax.scatter(bs_RNA[0], bs_RNA[1], bs_RNA[2], c='blue', label='oxRNA base site')
#ax.scatter(ss_RNA[0], ss_RNA[1], ss_RNA[2], c='forestgreen', label='oxRNA stack site')
ax.scatter(bbs_RNA[0], bbs_RNA[1], bbs_RNA[2], c='red', label='oxRNA back site')
ax.set_aspect('equal')

ax.legend()

# Alignment test zone

In [14]:
from oxDNA_analysis_tools.UTILS.utils import kabsch_align
from scipy.spatial.transform import Rotation
import timeit
points = np.random.rand(15000, 3)
ref = np.random.rand(15000, 3)
points_mean = np.mean(points, axis=0)
ref_mean = np.mean(ref, axis=0)

# Assert we calculated the same thing
my_impl = kabsch_align(points, ref)
rot, rssd = Rotation.align_vectors(ref - ref_mean, points - points_mean)
sp_impl = rot.apply(points - points_mean) + ref_mean
np.testing.assert_almost_equal(my_impl, sp_impl)
kabsch_align(points, ref, inplace=True)
np.testing.assert_almost_equal(points, sp_impl - ref_mean)

In [15]:
N = 10000
points = np.random.rand(15000, 3)
ref = np.random.rand(15000, 3)
points_mean = np.mean(points, axis=0)
ref_mean = np.mean(ref, axis=0)

# Time it
print("Big array")
print("Scipy:", timeit.timeit('rot, rssd = Rotation.align_vectors(ref - ref_mean, points - points_mean); rot.apply(points)', number=N, setup="from __main__ import Rotation, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')
print("Copy: ", timeit.timeit('kabsch_align(points, ref)', number=N, setup="from __main__ import kabsch_align, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')
print("Inpl: ", timeit.timeit('kabsch_align(points, ref, True)', number=N, setup="from __main__ import kabsch_align, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')

points = np.random.rand(10, 3)
ref = np.random.rand(10, 3)
points_mean = np.mean(points, axis=0)
ref_mean = np.mean(ref, axis=0)
print("lil array")
print("Scipy:", timeit.timeit('rot, rssd = Rotation.align_vectors(ref - ref_mean, points - points_mean); rot.apply(points)', number=N, setup="from __main__ import Rotation, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')
print("Copy: ", timeit.timeit('kabsch_align(points, ref)', number=N, setup="from __main__ import kabsch_align, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')
print("Inpl: ", timeit.timeit('kabsch_align(points, ref, True)', number=N, setup="from __main__ import kabsch_align, points, ref, points_mean, ref_mean") / N * 1000000, 'µs')

Big array
Scipy: 680.1441499963403 µs
Copy:  655.7241791975684 µs
Inpl:  641.3507333025336 µs
lil array
Scipy: 54.42207088926807 µs
Copy:  22.94562499737367 µs
Inpl:  22.99432089785114 µs
